# CoraML Facial-Walk Experiment

Train the transformer on facial walks, generate synthetic walks, reconstruct a graph, and compare link-prediction and graph statistics against the true CoraML graph.

## Ready To Run On T4

- This notebook is ready for a `Colab T4 GPU` runtime, not a TPU runtime.
- The training code uses `PyTorch + Hugging Face GPT-2`, so the intended accelerator is `cuda`.
- The graph split is `10% validation + 5% test`, while keeping the training graph connected.
- This cloud config targets about `1M` training chunks using `num_sign_configs=600`, `context_size=32`, and `stride=8`.
- Checkpoints are written every epoch to `checkpoints/coraml_t4_run`, and the final model is written to `checkpoints/coraml_t4_run/final`.
- If the runtime disconnects, rerunning the training cell with `resume_from_latest=True` resumes from the newest saved epoch.


In [ ]:
import sys
import importlib
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import facialgen
import facialgen.data as data_mod
import facialgen.early_stopping as early_stopping
import facialgen.evaluation as evaluation
import facialgen.models as models
import facialgen.train as train_mod

importlib.invalidate_caches()
importlib.reload(data_mod)
importlib.reload(early_stopping)
importlib.reload(evaluation)
importlib.reload(models)
importlib.reload(train_mod)
importlib.reload(facialgen)

from facialgen.data import CyclicFaceChunkDataset
from facialgen.early_stopping import (
    connected_link_prediction_split,
    edge_overlap_ratio,
    link_prediction_scores_from_walks,
    sample_model_walks,
)
from facialgen.evaluation import (
    compute_graph_statistics,
    reconstruct_graph_from_generated_walks,
)
from facialgen.models import FacialGen
from facialgen.train import build_training_objects, resolve_device, seed_everything, train_model

print('CyclicFaceChunkDataset from:', CyclicFaceChunkDataset.__module__)

args = SimpleNamespace(
    dataset_name='coraml',
    seed=2026,
    data_dir='data',
    num_sign_configs=600,
    sign_seed=2026,
    epoch_seed=99,
    context_size=32,
    stride=8,
    batch_size=64,
    epochs=20,
    lr=3e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    num_workers=0,
    device='auto',
    n_layer=1,
    n_head=4,
    n_embd=128,
    dropout=0.0,
    save_dir='checkpoints/coraml_t4_run',
    resume_from_latest=True,
    log_every=20,
    early_stop_mode='val',
    early_stop_patience=5,
    early_stop_min_delta=0.0,
    val_fraction=0.10,
    test_fraction=0.05,
    split_seed=123,
    eval_generated_walks=1_000_000,
    eval_max_length=None,
    target_edge_overlap=0.5,
    use_link_prediction_split=True,
)

checkpoint_dir = None
num_generated_graphs = 1
final_generated_walks = 500_000
final_max_length = args.context_size
generation_batch_size = 256
reconstruction_seed = 777

seed_everything(args.seed)
print(f"seed = {args.seed}")

device = resolve_device(args.device)
print(f"dataset = {args.dataset_name}")
print(f"context_size = {args.context_size}, stride = {args.stride}")
print(f"FAST baseline config for CoraML: n_layer = {args.n_layer}")
print(f"checkpoint_dir = {args.save_dir}")
print(f"resume_from_latest = {args.resume_from_latest}")
print('device =', device)


CyclicFaceChunkDataset from: facialgen.data
seed = 2026
dataset = coraml
context_size = 32, stride = 8
FAST baseline config for CoraML: n_layer = 1
checkpoint_dir = checkpoints/coraml_t4_run
resume_from_latest = True
device = mps


## Colab Setup

If you run this notebook in Google Colab, this cell mounts Drive for persistent checkpoints and rewrites `args.save_dir` into Drive. Locally it is a no-op.


In [ ]:
running_in_colab = False
drive_root = None

try:
    from google.colab import drive  # type: ignore
    running_in_colab = True
except ImportError:
    drive = None

if running_in_colab:
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')
    args.save_dir = str(drive_root / 'facialgen_checkpoints' / 'coraml_t4_run')
    Path(args.save_dir).mkdir(parents=True, exist_ok=True)
    print('Mounted Google Drive.')
    print('save_dir =', args.save_dir)
else:
    print('Not running in Colab; leaving save_dir unchanged.')
    print('save_dir =', args.save_dir)


## Reproducibility

Rerunning the import/config cell above now reloads `facialgen` and reseeds Python, NumPy, and PyTorch through `seed_everything(args.seed)`.


In [38]:
print(f"seed = {args.seed}")
print(f"dataset = {args.dataset_name}")
print(f"context_size = {args.context_size}, stride = {args.stride}")
print(f"num_sign_configs = {args.num_sign_configs}")
print(f"eval_generated_walks = {args.eval_generated_walks:,}")
print(f"final_generated_walks = {final_generated_walks:,}")
print(f"save_dir = {args.save_dir}")
print(f"resume_from_latest = {args.resume_from_latest}")


seed = 2026
dataset = coraml
context_size = 32, stride = 8
num_sign_configs = 10
eval_generated_walks = 1,000,000
final_generated_walks = 500,000
save_dir = checkpoints/coraml_t4_run
resume_from_latest = True


In [37]:
chunk_ds_preview, loader_preview, model_preview, eval_info_preview = build_training_objects(args)
print(f"live stride = {chunk_ds_preview.stride}")
print(f"num full face sequences = {len(chunk_ds_preview.face_dataset)}")
print(f"num chunk samples = {len(chunk_ds_preview)}")


Using connected train split for VAL early stopping: train_edges=6784, val_edges=798, test_edges=399
Dataset: coraml
LCC nodes: 2810
Full face sequences: 1394
Chunk samples @ T=32: 17294
Chunk stride: 8
Vocab: 2813 (vertices + BOS + EOS + PAD)
live stride = 8
num full face sequences = 1394
num chunk samples = 17294


In [34]:
if checkpoint_dir is None:
    model, eval_info, history = train_model(args)
else:
    _, _, _, eval_info = build_training_objects(args)
    model = FacialGen.from_pretrained(checkpoint_dir)
    history = []

model.to(device)
print(type(model).__name__)


Using connected train split for VAL early stopping: train_edges=6784, val_edges=798, test_edges=399
CoraML LCC nodes: 2810
Full face sequences: 14420
Chunk samples @ T=16: 175202
Chunk stride: 8
Vocab: 2813 (vertices + BOS + EOS + PAD)
Training on device: mps
Model config: layers=2, heads=2, embd=64, dropout=0.0


epoch 1/2:   0%|          | 0/21901 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['mean_nll'], marker='o', label='train NLL')
    axes[0].set_title('Training NLL by Epoch')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('mean_nll')
    axes[0].grid(True, alpha=0.3)
    if 'val_roc_auc' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_roc_auc'], marker='o', label='val ROC-AUC')
    if 'val_ap' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_ap'], marker='o', label='val AP')
    if 'val_score' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_score'], marker='o', label='val score')
    axes[1].set_title('Validation Metrics by Epoch')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('score')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

print('Final model checkpoint:', Path(args.save_dir) / 'final' if args.save_dir else 'not saved')


In [ ]:
reference_adj = eval_info['reference_adj']
reference_labels = eval_info['reference_labels']
num_nodes = int(eval_info['num_nodes'])
num_reference_edges = int(eval_info['num_reference_edges'])
lp_split = eval_info['link_prediction_split']
if lp_split is None:
    lp_split = connected_link_prediction_split(
        reference_adj,
        val_fraction=args.val_fraction,
        test_fraction=args.test_fraction,
        seed=args.split_seed,
    )

reference_stats = compute_graph_statistics(reference_adj, labels=reference_labels)

generated_results = []
generated_stats_rows = []

for graph_idx in range(num_generated_graphs):
    walks = sample_model_walks(
        model,
        num_samples=final_generated_walks,
        max_length=final_max_length,
        bos_token_id=int(eval_info['bos_token_id']),
        eos_token_id=int(eval_info['eos_token_id']),
        device=device,
        batch_size=generation_batch_size,
        show_progress=True,
        progress_desc=f'final sampling graph {graph_idx + 1}/{num_generated_graphs}',
    )

    A_hat, S = reconstruct_graph_from_generated_walks(
        walks,
        num_nodes=num_nodes,
        target_num_edges=num_reference_edges,
        seed=reconstruction_seed + graph_idx,
    )

    val_scores = link_prediction_scores_from_walks(
        walks,
        num_nodes=num_nodes,
        positive_edges=lp_split['val_edges'],
        negative_edges=lp_split['val_non_edges'],
    )
    test_scores = link_prediction_scores_from_walks(
        walks,
        num_nodes=num_nodes,
        positive_edges=lp_split['test_edges'],
        negative_edges=lp_split['test_non_edges'],
    )
    graph_stats = compute_graph_statistics(A_hat, labels=reference_labels)
    overlap = edge_overlap_ratio(A_hat, reference_adj)

    generated_results.append({
        'graph_id': graph_idx,
        'val_roc_auc': float(val_scores['roc_auc']),
        'val_ap': float(val_scores['average_precision']),
        'test_roc_auc': float(test_scores['roc_auc']),
        'test_ap': float(test_scores['average_precision']),
        'edge_overlap': float(overlap),
    })
    generated_stats_rows.append(graph_stats)

lp_table = pd.DataFrame(generated_results)
display(lp_table)

metric_names = list(reference_stats.keys())
stats_table = pd.DataFrame([
    {
        'metric': metric,
        'true_coraml': float(reference_stats[metric]),
        'generated_mean': float(np.nanmean([row[metric] for row in generated_stats_rows])),
        'abs_diff': abs(
            float(np.nanmean([row[metric] for row in generated_stats_rows]))
            - float(reference_stats[metric])
        ),
    }
    for metric in metric_names
])
display(stats_table)

if history:
    display(pd.DataFrame(history))
